# Basic Question & Answer Generation from a FileSet

This notebook demonstrates the simplest way to generate question-answer pairs from a FileSet. Documents are chunked into seeds, then questions and labels are generated in a single step using `QuestionAndLabelGenerator`.

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [1]:
%pip install lightningrod-ai python-dotenv -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [ ]:
fileset_id = "PASTE_YOUR_FILESET_ID_HERE"

## Configure the Question Pipeline

- **`FileSetSeedGenerator`** chunks the documents in your FileSet into seeds (text passages)
- **`QuestionAndLabelGenerator`** generates questions and answers in a single step — the simplest way to produce labeled Q&A pairs

In [4]:
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    QuestionAndLabelGenerator,
    FreeResponseAnswerType,
)

answer_type = FreeResponseAnswerType()

pipeline = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionAndLabelGenerator(
        questions_per_seed=2,
        answer_type=answer_type,
        instructions=(
            "Generate questions about the financial metrics, business events, "
            "and forward guidance in these quarterly investor reports. Questions should be "
            "specific and verifiable from the report content."
        ),
    ),
)

## Run the Pipeline

In [5]:
dataset = lr.transforms.run(
    pipeline,
    max_questions=10,
    name="FileSet - Basic QA",
)
print(f"Dataset: {dataset.id}")
print(f"Rows: {dataset.num_rows}")

/usr/local/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Dataset: 84fddcb3-19df-4298-b736-9dd116ecb855
Rows: 10


> **Note:** This can take a few minutes to complete processing.

## View the Results

In [6]:
%pip install pandas -q

from IPython.display import clear_output
clear_output()

In [7]:
import pandas as pd

samples = dataset.download()
rows = dataset.flattened(answer_type)
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples ({dataset.valid_count() / dataset.num_rows * 100:.1f}% valid)\n")

cols = ["question_text", "label", "label_confidence", "is_valid"]
df[[c for c in cols if c in df.columns]]

Generated 10 samples (100.0% valid)



,question_text,label,label_confidence
0,What is the maximum budget APEX Technologies I...,Up to $1.2 billion.,1.0
1,How much does APEX Technologies Inc. plan to i...,"The company plans to invest $400 million, whic...",1.0
2,What is APEX Technologies' expected revenue ra...,The expected revenue range for Q3 2024 is $2.3...,1.0
3,What was the year-over-year revenue growth per...,Revenue growth was 12% YoY and the operating m...,1.0
4,"According to the Q1 2025 report, when does APE...",APEX Technologies is targeting general availab...,1.0
5,What was the total acquisition cost of CyberSh...,The CyberShield acquisition cost was $980 mill...,1.0
6,What is the expected acquisition price and clo...,The acquisition is valued at $980 million and ...,1.0
7,When does APEX Technologies Inc. management pr...,Management projects the acquisition will be ac...,1.0
8,"According to the Q3 2024 report, when is the N...",The data center is 85% complete and is expecte...,1.0
9,Regarding APEX Technologies' infrastructure pr...,The data center expansion costs $400 million a...,1.0
